In [1]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
# Fix all seeds
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")
print(f"Seed       : {SEED} ✅")

TensorFlow : 2.20.0
Keras      : 3.13.2
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seed       : 42 ✅


In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/cats_vs_dogs_classifier"
MODEL_DIR  = Path(f"{DRIVE_PATH}/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SHAPE = (224, 224, 3)

print(f"Model weights will be saved to: {MODEL_DIR}")

Mounted at /content/drive
Model weights will be saved to: /content/drive/MyDrive/cats_vs_dogs_classifier/models


In [5]:
def build_custom_cnn(input_shape=INPUT_SHAPE):
    inputs = keras.Input(shape=input_shape, name="input")

    # ── Block 1: learns low-level features (edges, colours) ──────────────────
    x = layers.Conv2D(32, (3, 3), padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)       # 224×224 → 112×112
    x = layers.Dropout(0.25)(x)

    # ── Block 2: learns mid-level features (fur texture, eye shape) ──────────
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)       # 112×112 → 56×56
    x = layers.Dropout(0.25)(x)

    # ── Block 3: learns high-level features (face structure, body shape) ─────
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)       # 56×56 → 28×28
    x = layers.Dropout(0.25)(x)

    # ── Classification head ───────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D()(x)   # (28,28,128) → (128,)
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)
    # sigmoid → probability between 0.0 and 1.0
    # < 0.5 = Cat     >= 0.5 = Dog

    return keras.Model(inputs, outputs, name="custom_cnn")


cnn = build_custom_cnn()
cnn.summary()

Model: "custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 323,105 (1.23 MB)

 Trainable params: 321,697 (1.23 MB)

 Non-trainable params: 1,408 (5.50 KB)

In [6]:
total     = cnn.count_params()
trainable = sum(np.prod(v.shape) for v in cnn.trainable_variables)

print(f"Total parameters    : {total:,}")
print(f"Trainable           : {trainable:,}  (100% — trained from scratch)")
print(f"All params are new  : yes — no pretrained weights")
print()
print("Custom CNN built ✅")

Total parameters    : 323,105
Trainable           : 321,697  (100% — trained from scratch)
All params are new  : yes — no pretrained weights

Custom CNN built ✅


In [7]:
def build_transfer_model(base_model, name, dropout=0.5):
    """
    Wrap any pretrained base with our classification head.
    Base is frozen here (Phase 1 ready).
    Unfreezing for Phase 2 is done in training.ipynb.
    """
    base_model.trainable = False   # Phase 1: freeze the entire base

    inputs  = keras.Input(shape=INPUT_SHAPE, name="input")
    x       = base_model(inputs, training=False)
    # training=False keeps BatchNorm layers in inference mode
    # even when we call model.fit() — important for frozen bases

    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(dropout)(x)
    x       = layers.Dense(256, activation="relu", name="fc1")(x)
    x       = layers.Dropout(dropout * 0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    return keras.Model(inputs, outputs, name=name)


def count_params(model):
    total     = model.count_params()
    trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
    frozen    = total - trainable
    return total, trainable, frozen


print("Transfer model builder ready ✅")

Transfer model builder ready ✅


In [8]:
vgg16_base = keras.applications.VGG16(
    include_top  = False,          # remove the original 1000-class head
    weights      = "imagenet",     # download pretrained ImageNet weights
    input_shape  = INPUT_SHAPE,
)

vgg16 = build_transfer_model(vgg16_base, name="vgg16_transfer")

total, trainable, frozen = count_params(vgg16)
print("VGG16 Transfer Model ✅")
print(f"  Total parameters    : {total:,}")
print(f"  Trainable (Phase 1) : {trainable:,}   (head only)")
print(f"  Frozen              : {frozen:,}  (entire base)")
print()
print("Phase 2 unfreezing is done in training.ipynb")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
VGG16 Transfer Model ✅
  Total parameters    : 14,848,321
  Trainable (Phase 1) : 132,609   (head only)
  Frozen              : 14,715,712  (entire base)

Phase 2 unfreezing is done in training.ipynb


In [9]:
# Show what the base learned — visualise VGG16 layer names
print("VGG16 base layers (top 10):")
for i, layer in enumerate(vgg16_base.layers[-10:]):
    layer_num = len(vgg16_base.layers) - 10 + i
    print(f"  [{layer_num:>2}] {layer.name:<25} trainable={layer.trainable}")
print()
print("All set to trainable=False for Phase 1")
print("In Phase 2 we unfreeze the last 20 of these layers")

VGG16 base layers (top 10):
  [ 9] block3_conv3              trainable=False
  [10] block3_pool               trainable=False
  [11] block4_conv1              trainable=False
  [12] block4_conv2              trainable=False
  [13] block4_conv3              trainable=False
  [14] block4_pool               trainable=False
  [15] block5_conv1              trainable=False
  [16] block5_conv2              trainable=False
  [17] block5_conv3              trainable=False
  [18] block5_pool               trainable=False

All set to trainable=False for Phase 1
In Phase 2 we unfreeze the last 20 of these layers


Model 3


In [10]:
resnet_base = keras.applications.ResNet50(
    include_top = False,
    weights     = "imagenet",
    input_shape = INPUT_SHAPE,
)

resnet50 = build_transfer_model(resnet_base, name="resnet50_transfer")

total, trainable, frozen = count_params(resnet50)
print("ResNet50 Transfer Model ✅")
print(f"  Total parameters    : {total:,}")
print(f"  Trainable (Phase 1) : {trainable:,}   (head only)")
print(f"  Frozen              : {frozen:,}  (entire base)")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
ResNet50 Transfer Model ✅
  Total parameters    : 24,120,705
  Trainable (Phase 1) : 528,897   (head only)
  Frozen              : 23,591,808  (entire base)


# Model 4

In [11]:
effnet_base = keras.applications.EfficientNetB0(
    include_top = False,
    weights     = "imagenet",
    input_shape = INPUT_SHAPE,
)

efficientnet = build_transfer_model(
    effnet_base,
    name    = "efficientnetb0_transfer",
    dropout = 0.4,    # slightly lower dropout — EfficientNet already has built-in regularisation
)

total, trainable, frozen = count_params(efficientnet)
print("EfficientNetB0 Transfer Model ✅")
print(f"  Total parameters    : {total:,}")
print(f"  Trainable (Phase 1) : {trainable:,}   (head only)")
print(f"  Frozen              : {frozen:,}  (entire base)")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
EfficientNetB0 Transfer Model ✅
  Total parameters    : 4,382,884
  Trainable (Phase 1) : 330,753   (head only)
  Frozen              : 4,052,131  (entire base)


## Unfreeze the top n layers of the base model for Phase 2 fine-tuning.


In [12]:
def unfreeze_top_layers(model, n=20):
    """
    Unfreeze the top n layers of the base model for Phase 2 fine-tuning.

    Args:
        model : the full transfer learning model (base + head)
        n     : number of base layers to unfreeze from the top

    Why only top layers?
        Bottom layers detect universal features (edges, basic shapes).
        These are the same in almost every image — no need to change them.
        Top layers detect higher-level features more specific to our dataset.
    """
    base_model = model.layers[1]   # base is always layer index 1 in our wrapper
    base_model.trainable = True

    # Freeze everything except the top n layers
    for layer in base_model.layers[:-n]:
        layer.trainable = False

    total_base    = len(base_model.layers)
    now_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
    now_frozen    = model.count_params() - now_trainable

    print(f"  Base layers total   : {total_base}")
    print(f"  Layers unfrozen     : top {n}")
    print(f"  Trainable params    : {now_trainable:,}")
    print(f"  Frozen params       : {now_frozen:,}")
    print(f"  Use LR = 1e-5 now  (NOT 1e-3 — that would destroy pretrained features)")


# Demo: show what happens when we unfreeze VGG16
print("Before Phase 2 — VGG16:")
_, t_before, _ = count_params(vgg16)
print(f"  Trainable: {t_before:,}")

print()
print("After unfreeze_top_layers(vgg16, n=20):")
unfreeze_top_layers(vgg16, n=20)

# Re-freeze for this notebook (training.ipynb will do the real unfreezing)
vgg16.layers[1].trainable = False
for layer in vgg16.layers[1].layers:
    layer.trainable = False
print()
print("(Re-frozen for this notebook — training.ipynb handles the real Phase 2)")

Before Phase 2 — VGG16:
  Trainable: 132,609

After unfreeze_top_layers(vgg16, n=20):
  Base layers total   : 19
  Layers unfrozen     : top 20
  Trainable params    : 14,847,297
  Frozen params       : 1,024
  Use LR = 1e-5 now  (NOT 1e-3 — that would destroy pretrained features)

(Re-frozen for this notebook — training.ipynb handles the real Phase 2)


# Model Comparsion

In [13]:
models_info = [
    ("Custom CNN",       cnn,         "From scratch",   "~78%",  "~80%",  "~2M"),
    ("VGG16",            vgg16,       "Transfer + FT",  "~97%",  "~98%",  "138M"),
    ("ResNet50",         resnet50,    "Transfer + FT",  "~95%",  "~96%",  "25M"),
    ("EfficientNetB0",   efficientnet,"Transfer + FT",  "~94%",  "~95%",  "5.3M"),
]

print(f"  {'Model':<18} {'Params':>12}  {'Strategy':<15} {'Test Acc':>9} {'Val Acc':>9}")
print(f"  {'-'*18} {'-'*12}  {'-'*15} {'-'*9} {'-'*9}")

for name, model, strategy, test_acc, val_acc, params in models_info:
    actual_params = f"{model.count_params():,}"
    star = " ⭐" if name == "VGG16" else ""
    print(f"  {name+star:<20} {actual_params:>12}  {strategy:<15} {test_acc:>9} {val_acc:>9}")

print()
print("  ⭐ = production model (best accuracy)")
print("  ⚡ = EfficientNetB0 best for deployment (smallest)")

  Model                    Params  Strategy         Test Acc   Val Acc
  ------------------ ------------  --------------- --------- ---------
  Custom CNN                323,105  From scratch         ~78%      ~80%
  VGG16 ⭐                14,848,321  Transfer + FT        ~97%      ~98%
  ResNet50               24,120,705  Transfer + FT        ~95%      ~96%
  EfficientNetB0          4,382,884  Transfer + FT        ~94%      ~95%

  ⭐ = production model (best accuracy)
  ⚡ = EfficientNetB0 best for deployment (smallest)


# Verify all models

In [14]:
# Create one fake batch: 2 images of 224x224x3
dummy_input = tf.random.normal((2, 224, 224, 3))

print("Running dummy forward pass through all 4 models...\n")

all_models = [
    ("Custom CNN",     cnn),
    ("VGG16",          vgg16),
    ("ResNet50",       resnet50),
    ("EfficientNetB0", efficientnet),
]

for name, model in all_models:
    output = model(dummy_input, training=False)
    print(f"  {name:<18}")
    print(f"    Input  : {dummy_input.shape}")
    print(f"    Output : {output.shape}   <- (batch=2, probability=1)")
    print(f"    Values : {output.numpy().flatten().tolist()}")
    print(f"    Range  : {output.numpy().min():.4f} - {output.numpy().max():.4f}  (sigmoid: always 0-1)")
    print()

print("All models produce correct output shape (batch, 1) ✅")
print("Values are between 0 and 1 (sigmoid output) ✅")
print("< 0.5 = Cat     >= 0.5 = Dog")

Running dummy forward pass through all 4 models...

  Custom CNN        
    Input  : (2, 224, 224, 3)
    Output : (2, 1)   <- (batch=2, probability=1)
    Values : [0.4983558654785156, 0.49852225184440613]
    Range  : 0.4984 - 0.4985  (sigmoid: always 0-1)

  VGG16             
    Input  : (2, 224, 224, 3)
    Output : (2, 1)   <- (batch=2, probability=1)
    Values : [0.3508622348308563, 0.35879766941070557]
    Range  : 0.3509 - 0.3588  (sigmoid: always 0-1)

  ResNet50          
    Input  : (2, 224, 224, 3)
    Output : (2, 1)   <- (batch=2, probability=1)
    Values : [0.4867795705795288, 0.47856244444847107]
    Range  : 0.4786 - 0.4868  (sigmoid: always 0-1)

  EfficientNetB0    
    Input  : (2, 224, 224, 3)
    Output : (2, 1)   <- (batch=2, probability=1)
    Values : [0.3857002258300781, 0.3836230933666229]
    Range  : 0.3836 - 0.3857  (sigmoid: always 0-1)

All models produce correct output shape (batch, 1) ✅
Values are between 0 and 1 (sigmoid output) ✅
< 0.5 = Cat   

# Save model definition

In [15]:
import json

# Save model configs (architecture only, no weights)
configs = {}
for name, model in all_models:
    save_name = name.lower().replace(" ", "_")
    config_path = MODEL_DIR / f"{save_name}_config.json"
    with open(config_path, "w") as f:
        json.dump(model.get_config(), f, indent=2, default=str)
    configs[name] = str(config_path)
    print(f"  Saved config: {config_path.name}")

print()
print("Architecture configs saved ✅")
print("training.ipynb will rebuild the models and train them.")

  Saved config: custom_cnn_config.json
  Saved config: vgg16_config.json
  Saved config: resnet50_config.json
  Saved config: efficientnetb0_config.json

Architecture configs saved ✅
training.ipynb will rebuild the models and train them.
